# Comparación pareada: LightGBM sin RF vs LightGBM con RF

Este notebook compara ambos modelos usando exactamente las mismas semillas. La unidad de análisis es cada semilla, por lo que las ganancias deben estar en el mismo orden en ambos vectores.


In [ ]:
# Limpieza y librerías
rm(list = ls(all.names = TRUE))
gc(full = TRUE, verbose = FALSE)

require("data.table")
require("ggplot2")


## 1. Cargar resultados

Reemplazá únicamente las ganancias cuando tengas nuevas corridas. No cambies el orden de las semillas.


In [ ]:
# Semillas apareadas
semillas <- c(157627, 386003, 548177, 874127, 936931)

# Ganancias observadas con las mismas semillas
# El orden debe coincidir exactamente con el vector semillas.
gan_con_rf <- c(595, 581, 586, 585, 571)
gan_sin_rf <- c(598, 579, 612, 589, 598)

stopifnot(
  length(semillas) == length(gan_con_rf),
  length(semillas) == length(gan_sin_rf),
  !anyNA(gan_con_rf),
  !anyNA(gan_sin_rf)
)

tb_resultados <- data.table(
  semilla = semillas,
  con_RF = gan_con_rf,
  sin_RF = gan_sin_rf
)

tb_resultados[, diferencia_sin_menos_con := sin_RF - con_RF]
tb_resultados


## 2. Resumen descriptivo

Una diferencia positiva significa que el modelo **sin RF** obtuvo mayor ganancia en esa semilla.


In [ ]:
resumen <- data.table(
  modelo = c("LightGBM con RF", "LightGBM sin RF"),
  media = c(mean(gan_con_rf), mean(gan_sin_rf)),
  mediana = c(median(gan_con_rf), median(gan_sin_rf)),
  desvio = c(sd(gan_con_rf), sd(gan_sin_rf)),
  minimo = c(min(gan_con_rf), min(gan_sin_rf)),
  maximo = c(max(gan_con_rf), max(gan_sin_rf))
)

resumen

cat("Diferencia promedio (sin RF - con RF):", mean(tb_resultados$diferencia_sin_menos_con), "\n")
cat("Diferencia mediana (sin RF - con RF):", median(tb_resultados$diferencia_sin_menos_con), "\n")
cat("Semillas ganadas por sin RF:", sum(tb_resultados$diferencia_sin_menos_con > 0), "de", nrow(tb_resultados), "\n")
cat("Mejora porcentual promedio respecto de con RF:",
    round(100 * (mean(gan_sin_rf) - mean(gan_con_rf)) / mean(gan_con_rf), 2), "%\n")


## 3. Wilcoxon pareado

Se informan dos pruebas:

- **Bilateral:** detecta cualquier diferencia entre los modelos.
- **Unilateral:** prueba específicamente si la ganancia de LightGBM sin RF es mayor que la de LightGBM con RF.

La prueba unilateral debe definirse antes de mirar los resultados si forma parte de la hipótesis principal.


In [ ]:
wilcox_bilateral <- wilcox.test(
  gan_sin_rf,
  gan_con_rf,
  paired = TRUE,
  alternative = "two.sided",
  exact = TRUE,
  conf.int = TRUE
)

wilcox_sin_mayor <- wilcox.test(
  gan_sin_rf,
  gan_con_rf,
  paired = TRUE,
  alternative = "greater",
  exact = TRUE,
  conf.int = TRUE
)

cat("\nWilcoxon bilateral:\n")
print(wilcox_bilateral)

cat("\nWilcoxon unilateral: sin RF > con RF:\n")
print(wilcox_sin_mayor)


## 4. Tabla final de interpretación


In [ ]:
alpha <- 0.05

conclusion <- data.table(
  comparacion = c("Diferencia entre modelos", "Sin RF mayor que con RF"),
  alternativa = c("two.sided", "greater"),
  p_value = c(wilcox_bilateral$p.value, wilcox_sin_mayor$p.value)
)

conclusion[, significativo_0_05 := p_value < alpha]
conclusion[, interpretacion := fifelse(
  significativo_0_05,
  "Hay evidencia estadística al 5%",
  "No hay evidencia estadística suficiente al 5%"
)]

conclusion


## 5. Gráfico pareado

Las líneas unen los resultados obtenidos con la misma semilla. Esto permite ver la dirección y magnitud del cambio en cada corrida.


In [ ]:
tb_largo <- melt(
  tb_resultados,
  id.vars = "semilla",
  measure.vars = c("con_RF", "sin_RF"),
  variable.name = "modelo",
  value.name = "ganancia"
)

tb_largo[, modelo := factor(
  modelo,
  levels = c("con_RF", "sin_RF"),
  labels = c("LightGBM con RF", "LightGBM sin RF")
)]

grafico_pareado <- ggplot(
  tb_largo,
  aes(x = modelo, y = ganancia, group = semilla)
) +
  geom_line(alpha = 0.55) +
  geom_point(aes(shape = factor(semilla)), size = 3) +
  stat_summary(
    aes(group = 1),
    fun = mean,
    geom = "point",
    size = 4
  ) +
  labs(
    title = "Comparación pareada de ganancias",
    subtitle = paste0("Mismas semillas | n = ", length(semillas)),
    x = NULL,
    y = "Ganancia",
    shape = "Semilla"
  ) +
  theme_bw() +
  theme(legend.position = "right")

print(grafico_pareado)

# Para guardar el gráfico, descomentar:
# ggsave("comparacion_pareada_LightGBM_RF.pdf", grafico_pareado, width = 8, height = 6)


## 6. Exportar resultados


In [ ]:
fwrite(tb_resultados, "resultados_pareados_lightgbm.csv")
fwrite(resumen, "resumen_modelos_lightgbm.csv")
fwrite(conclusion, "wilcoxon_lightgbm.csv")

cat("Archivos exportados correctamente.\n")
